# Cocogoat 2026 - Task 2: VAE + latent world model (64x64 video)

Per-frame beta-VAE (ConvNeXt depthwise blocks, PixelUnshuffle with DC-AE shortcuts, bottleneck attention, EQ-VAE regulariser, adversarial fine-tuning) + a space-time transformer world model with EMA memory that predicts the next frame's latent. Two datasets: Colored Moving MNIST (controlled dynamics, the first rung of your ViVAE curriculum) and your `Shelterdataset.npz`. Budget: ~1.5 h per dataset. Outputs: `MyDrive/Cocogoat/runs/task2/<dataset>/{vae,wm}/`.

**How to run:** `Runtime -> Change runtime type -> G4 (RTX PRO 6000)`, then `Runtime -> Run all`.
Keep this tab open (Colab Pro stops idle/closed sessions). Everything is written to
`MyDrive/Cocogoat/runs/...`; if Colab disconnects, just *Run all* again - training resumes from the last checkpoint.
You do not need to inspect results: metrics, sample grids and logs are saved to Drive for analysis.

In [ ]:
# 1) Mount Google Drive (outputs + datasets live there)
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 2) Get / update the code (public repo, branch claude/cocogoat-revival-glfnmz)
import os, subprocess
REPO, BRANCH = "https://github.com/Martyn0324/Cocogoat.git", "claude/cocogoat-revival-glfnmz"
if not os.path.exists('/content/Cocogoat/.git'):
    subprocess.run(['git', 'clone', '-b', BRANCH, '--depth', '1', REPO, '/content/Cocogoat'], check=True)
else:
    subprocess.run(['git', '-C', '/content/Cocogoat', 'fetch', '--depth', '1', 'origin', BRANCH], check=True)
    subprocess.run(['git', '-C', '/content/Cocogoat', 'reset', '--hard', 'FETCH_HEAD'], check=True)
print(subprocess.run(['git', '-C', '/content/Cocogoat', 'log', '-1', '--oneline'], capture_output=True, text=True).stdout)
os.chdir('/content/Cocogoat')

In [ ]:
# 3) Extra packages (torch/torchvision/transformers come with Colab)
!pip -q install -r /content/Cocogoat/requirements-colab.txt
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
# Colab Secrets are only readable from the notebook kernel: export HF_TOKEN for the training subprocesses
try:
    from google.colab import userdata
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
    print('HF_TOKEN exported')
except Exception as exc:
    print('No HF_TOKEN secret (only needed for gated Hugging Face datasets):', type(exc).__name__)
os.environ['PYTHONUNBUFFERED'] = '1'

In [ ]:
# 4) GPU smoke test (tiny models, a few steps)
!python -m cocogoat.tasks.task2_world --smoke --dataset mmnist --run_root /content/drive/MyDrive/Cocogoat/runs/task2_smoke --force

In [ ]:
# 5) Colored Moving MNIST: VAE (45 min) then world model (45 min); resumes automatically
!python -m cocogoat.tasks.task2_world --dataset mmnist

In [ ]:
# 6) Shelter (your anime music-video frames): VAE (45 min) then world model (45 min)
!python -m cocogoat.tasks.task2_world --dataset shelter

In [ ]:
# 7) Results
from IPython.display import Image, Markdown, display
for ds in ['mmnist', 'shelter']:
    run = f'/content/drive/MyDrive/Cocogoat/runs/task2/{ds}'
    try:
        display(Markdown(f'## {ds}\n' + open(f'{run}/wm/report.md').read()))
        display(Image(f'{run}/vae/samples/recon_final.png'))
        display(Image(f'{run}/wm/samples/rollout_final.png'))
    except FileNotFoundError as exc:
        print(ds, 'not finished:', exc)